# Simple example of using pipelines

### 1. Import everything

In [73]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix , accuracy_score , precision_score,recall_score,f1_score

### 2. Create or Use Dataset

In [6]:
df = pd.DataFrame({
    "Age": [25, 30, 45, 35, 22, 40],
    "Salary": [50000, 60000, 80000, 65000, 48000, 75000],
    "Gender": ["Male", "Female", "Female", "Male", "Female", "Male"],
    "City": ["Pune", "Mumbai", "Pune", "Delhi", "Delhi", "Mumbai"],
    "Bought": [0, 1, 1, 0, 0, 1]
})

In [13]:
df

,Age,Salary,Gender,City,Bought
0,25,50000,Male,Pune,0
1,30,60000,Female,Mumbai,1
2,45,80000,Female,Pune,1
3,35,65000,Male,Delhi,0
4,22,48000,Female,Delhi,0
5,40,75000,Male,Mumbai,1


### 3. Separate Features and Target

In [15]:
X = df.drop("Bought", axis=1)
y = df["Bought"]

### 4. Separate column types 

In [47]:
num_cols = ["Age", "Salary"]
cat_cols = ["Gender", "City"]

## 5. Create Sub-Pipelines

### Numerical Pipeline

In [51]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

### Categorical Pipeline

In [54]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder())
])

## 6. Create Column Transformer

In [58]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

## 7. Create the FINAL PIPELINE (Preprocessing + Model)

In [61]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])

### 8. Train - Test Split

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 9. Train the model and visualize the final pipeline

In [67]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder())]),
                                                  ['Gender', 'City'])])),
                ('model', LogisticRegression())])

### 10. Prediction of Model

In [70]:
y_pred = pipeline.predict(X_test)

### 11. Hyperparameter Tuning

In [ ]:
param_grid = {
    "model__C": [0.1, 1, 10]
}

grid = GridSearchCV(pipeline, param_grid, cv=2)
grid.fit(X_train, y_train)

print("Best Parameter:", grid.best_params_)

### 12. Accessing Named Steps

In [95]:
print(pipeline.named_steps)

{'preprocessor': ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Salary']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder', OneHotEncoder())]),
                                 ['Gender', 'City'])]), 'model': LogisticRegression()}


In [97]:
pipeline.named_steps["model"]

LogisticRegression()

In [99]:
pipeline.named_steps["preprocessor"]

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Salary']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder', OneHotEncoder())]),
                                 ['Gender', 'City'])])

In [101]:
pipeline.named_steps["preprocessor"].named_transformers_["num"]

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

In [103]:
pipeline.named_steps["preprocessor"].named_transformers_["cat"]

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder())])